In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML

# Import data

In [2]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 0.5, "long_term": 0.1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,0.50,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.49,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.48,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.47,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.46,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

_Let $l_{u,i}$ denote the event that user $u$ has chosen to interact with item $i$ (user $u$ prefers item $i$). Then, we can let the probability of this event occurring be distributed according to a logistic function parameterized by the sum of the inner product of user and item latent factor vectors and user and item biases._
$$
p(l_{ui} | x_u, y_i, \beta_i, \beta_j) = \frac{\exp(x_u^Ty_i + \beta_u + \beta_i)}{1 + \exp(x_u^Ty_i + \beta_u + \beta_i)}
$$



In [3]:
def probability_user_item_interaction(user_latent: np.ndarray, item_latent: np.ndarray, user_bias: float, item_bias: float) -> float:
    """
    Computes the probability of a user interacting with an item.
    :param user_latent: The latent vector of the user.
    :param item_latent: The latent vector of the item.
    :param user_bias: The bias of the user.
    :param item_bias: The bias of the item.
    :return: The probability of the user interacting with the item.
    """
    return torch.exp(torch.dot(user_latent, item_latent) + user_bias + item_bias) / (1 + torch.exp(torch.dot(user_latent, item_latent) + user_bias + item_bias))

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"] == "top_track"]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]

,username,id,affinity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
0,jaslkh,75rqqKvzJCGv2oq9C4yFDt,0.500,0.630,0.374,0.2270,0.8610,0.000075,0.2940,0.3730,104.955,-9.007,196426,2023,63
1,jaslkh,2FYGZDfsAnNsrm1gVbyKnG,0.490,0.827,0.768,0.2650,0.7900,0.000024,0.4970,0.7340,99.988,-5.702,137533,2022,70
2,jaslkh,4kroNlz8BTfswE4M0i3YCh,0.480,0.583,0.297,0.4060,0.8790,0.000000,0.1270,0.2910,124.279,-11.273,162906,2022,61
3,jaslkh,2N3YZ075lq9z1ObaAiX6l1,0.470,0.651,0.327,0.4020,0.9280,0.000000,0.2250,0.5510,135.325,-10.070,89749,2022,54
4,jaslkh,2SiAcexM2p1yX6joESbehd,0.460,0.555,0.634,0.2730,0.1290,0.000002,0.1880,0.5550,170.228,-5.522,174044,2023,73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12318,dany,4GRMPDD3V6rM2BlmRdYCUJ,0.010,0.491,0.699,0.0924,0.1620,0.864000,0.1050,0.0762,130.325,-10.500,214000,2022,41
12319,dany,17Xof0GRZZfS7ZgjUJ27pH,0.008,0.271,0.503,0.0314,0.7500,0.000102,0.0941,0.0742,127.960,-4.142,241821,2020,50
12320,dany,2Rd4eJ4KwXQQn2sMSToyUM,0.006,0.319,0.236,0.0318,0.8980,0.000016,0.2720,0.2540,162.351,-9.644,156091,2021,6
12321,dany,4fPBB44eDH71YohayI4eKV,0.004,0.630,0.908,0.0326,0.0238,0.592000,0.1160,0.9680,120.522,-2.420,189226,2006,75


In [5]:
# Create the matrix dataset
class MatrixDataset:
    _matrix: np.ndarray

    @property
    def matrix(self) -> np.ndarray:
        return self._matrix

    def __init__(self, num_users: int, num_items: int):
        # A matrix of shape (num_users, num_items)
        self._matrix = np.zeros((num_users, num_items))
    
    def add_interaction(self, user_id: int, item_id: int, value: float):
        """
        Adds a new interaction to the matrix.
        :param user_id: The user id.
        :param item_id: The item id.
        :param value: The value of the interaction.
        """
        self._matrix[user_id, item_id] = value

    def fill_from_df(self, users: pd.Series, items: pd.Series, values: pd.Series):
        """
        Fills the matrix from a dataframe.
        :param df: The dataframe.
        :param user_col: The user column.
        :param item_col: The item column.
        :param value_col: The value column.
        """
        assert len(users) == len(items) == len(values), "The length of the users, items and values must be the same."

        for user, item, value in zip(users, items, values):
            self.add_interaction(user, item, value)
    
    def __str__(self):
        return str(self._matrix)

In [6]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())

In [7]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
matrix_mf.matrix.shape

(8, 923)

In [8]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [9]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

## Define the confidence of the entry of the matrix

In [10]:
def log_posterior(X: torch.Tensor, Y: torch.Tensor, beta_u: torch.Tensor, beta_i: torch.Tensor, R: torch.Tensor, alpha: float, lambd: float) -> torch.Tensor:
    term1 = alpha * R * (torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)
    term2 = (1 + alpha * R) * torch.log1p(torch.exp(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i))
    regularization = (lambd / 2) * (torch.norm(X, p=2) ** 2 + torch.norm(Y, p=2) ** 2)
    return torch.sum(term1 - term2) - regularization

def gradient(X: torch.Tensor, Y: torch.Tensor, beta_u: torch.Tensor, beta_i: torch.Tensor, R: torch.Tensor, alpha: float, lambd: float) -> torch.Tensor:
    exp_term = torch.exp(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)
    gradient_X = torch.matmul((alpha * R - (R / (1 + alpha * R)) * exp_term), Y) - lambd * X
    return gradient_X

def mpr(I: torch.Tensor, R: torch.Tensor) -> float:
    """
    Compute the Mean Percentile Ranking (MPR) using a sorted index matrix.

    :param I: Matrix of sorted indices of items for each user.
    :param R: Rating or interaction matrix.
    :return: MPR value.
    """
    num_users, num_items = R.shape
    total_interactions = torch.sum(R)

    # Initialize MPR
    mpr = 0.0

    # Iterate over each user
    for u in range(num_users):
        # Get the indices of the items in sorted order for this user
        sorted_indices = I[u]

        # Calculate the rank for each item
        for i in range(num_items):
            item_index = sorted_indices[i]
            rank = i / num_items  # Percentile rank
            mpr += R[u, item_index] * rank

    # Normalize by the total number of interactions
    mpr /= total_interactions

    return mpr

In [11]:
def adagrad_update(X: torch.Tensor, grad_X: torch.Tensor, grad_accumulator: torch.Tensor, gamma: float) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Perform AdaGrad update on the latent factor matrix X.

    :param X: Latent factor matrix (e.g., user or item matrix).
    :param grad_X: Gradient of the loss with respect to X.
    :param grad_accumulator: Accumulator for the sum of squares of past gradients.
    :param gamma: Step size parameter.
    :return: Updated latent factor matrix and updated accumulator.
    """
    # Update the accumulator with the square of the current gradient
    grad_accumulator += grad_X.pow(2)

    # Update the latent factor matrix
    X -= gamma * grad_X / torch.sqrt(grad_accumulator)

    return X, grad_accumulator

In [12]:
def predict_interaction(X, Y, beta_u, beta_i):
    """
    Predict the probability of interaction between a user and an item.

    :param x_u: User latent factor vector.
    :param y_i: Item latent factor vector.
    :param beta_u: User bias term.
    :param beta_i: Item bias term.
    :return: Probability of interaction.
    """
    return torch.sigmoid(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)

## The goal is to learn X, Y and B that maximize the likelihood of the log posterior

In [13]:
num_factors = 10        # Latent factor dimension
gamma = 0.01            # Learning rate
lambd = 0.1             # Regularization parameter
alpha = 40              # Scaling parameter
learning_rate = 0.1     # Learning rate
epochs = 100            # Number of epochs
mprs = []               # Record MPRs for each epoch

# Initialize user and item latent factor matrices and bias vectors
R = torch.tensor(matrix_mf.matrix, dtype=torch.float32)
X = torch.randn(num_users, num_factors, requires_grad=True)
Y = torch.randn(num_items, num_factors, requires_grad=True)
beta_u = torch.randn(num_users, requires_grad=True)
beta_i = torch.randn(num_items, requires_grad=True)

# Initialize the AdaGrad accumulators
grad_accumulator_X = torch.zeros_like(X)
grad_accumulator_Y = torch.zeros_like(Y)
grad_accumulator_beta_u = torch.zeros_like(beta_u)
grad_accumulator_beta_i = torch.zeros_like(beta_i)

# Training loop
for epoch in range(epochs):
    # Compute predictions for all user-item pairs
    predictions = predict_interaction(X, Y, beta_u, beta_i)

    # Calculate the log posterior (as the loss)
    loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)

    # Perform backpropagation
    loss.backward()

    # Perform parameter updates using AdaGrad
    with torch.no_grad():
        # Update user and item latent factors and biases
        X, grad_accumulator_X = adagrad_update(X, X.grad, grad_accumulator_X, learning_rate)
        Y, grad_accumulator_Y = adagrad_update(Y, Y.grad, grad_accumulator_Y, learning_rate)
        beta_u, grad_accumulator_beta_u = adagrad_update(beta_u, beta_u.grad, grad_accumulator_beta_u, learning_rate)
        beta_i, grad_accumulator_beta_i = adagrad_update(beta_i, beta_i.grad, grad_accumulator_beta_i, learning_rate)
        
        # Zero out gradients for the next iteration
        X.grad.zero_()
        Y.grad.zero_()
        beta_u.grad.zero_()
        beta_i.grad.zero_()

    # Compute and monitor MPR
    # Sort the predictions for each user to get ranking
    _, sorted_indices = torch.sort(predictions, descending=True, dim=1)
    mpr_value = mpr(sorted_indices, R)
    mprs.append(mpr_value.item())
    print(f"Epoch {epoch+1}/{epochs}, MPR: {mpr_value.item()}")

Epoch 1/100, MPR: 0.5386714935302734
Epoch 2/100, MPR: 0.47024133801460266
Epoch 3/100, MPR: 0.425534188747406
Epoch 4/100, MPR: 0.3877871334552765
Epoch 5/100, MPR: 0.3539358377456665
Epoch 6/100, MPR: 0.32409414649009705
Epoch 7/100, MPR: 0.2968372404575348
Epoch 8/100, MPR: 0.27222055196762085
Epoch 9/100, MPR: 0.24949146807193756
Epoch 10/100, MPR: 0.22957368195056915
Epoch 11/100, MPR: 0.2118302285671234
Epoch 12/100, MPR: 0.1956997662782669
Epoch 13/100, MPR: 0.18077567219734192
Epoch 14/100, MPR: 0.16776716709136963
Epoch 15/100, MPR: 0.15618635714054108
Epoch 16/100, MPR: 0.14592276513576508
Epoch 17/100, MPR: 0.13680221140384674
Epoch 18/100, MPR: 0.12849372625350952
Epoch 19/100, MPR: 0.12112963944673538
Epoch 20/100, MPR: 0.1143953949213028
Epoch 21/100, MPR: 0.10849222540855408
Epoch 22/100, MPR: 0.10284742712974548
Epoch 23/100, MPR: 0.09791957587003708
Epoch 24/100, MPR: 0.09342974424362183
Epoch 25/100, MPR: 0.08941124379634857
Epoch 26/100, MPR: 0.08588378131389618
Epoc

In [14]:
px.line(y=mprs, title="MPR over epochs")

In [15]:
user_latent = X.detach().numpy()
item_latent = Y.detach().numpy()

if num_factors <= 3:
    # Add the latent vectors to the dataframe
    df_matrix_mf["user_latent"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: user_latent[x]
    )
    df_matrix_mf["item_latent"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: item_latent[x]
    )

    # Add the biases to the dataframe
    df_matrix_mf["user_bias"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: beta_u[x].item()
    )
    df_matrix_mf["item_bias"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: beta_i[x].item()
    )

    for i in range(num_factors):
        df_matrix_mf[f"user_latent_{i}"] = df_matrix_mf["user_latent"].apply(lambda x: x[i])
        df_matrix_mf[f"latent_{i}"] = df_matrix_mf["item_latent"].apply(lambda x: x[i])

    fig = plotting.plot_latent_space(
        df_matrix_mf,
        color=df_matrix_mf["username"],
        text=df_matrix_mf["username"],
        title="User latent space",
        latent_columns=["latent_0", "latent_1", "latent_2"],
    )
    fig.show()

    df_user_latent = df_matrix_mf.drop_duplicates(subset=["username"])
    fig = plotting.plot_latent_space(
        df_user_latent,
        color=df_user_latent["username"],
        text=df_user_latent["username"],
        title="User latent space",
        latent_columns=["user_latent_0", "user_latent_1", "user_latent_2"],
    )
    fig.show()

In [19]:
# Choose a user
user = "michelle"
user_id = convert_to_ids([user], "username")[0]

# Make predictions for the user
predictions = predict_interaction(X, Y, beta_u, beta_i)
user_predictions = predictions[user_id]

# Sort all predictions in descending order
sorted_indices = torch.argsort(user_predictions, descending=True)

# Get the top N recommendations
N = 10
top_N = sorted_indices[:N].numpy()

# Get the top N recommendations in terms of the original item ids
top_N_items = retrieve_value_from_ids(top_N, "id")

# Get the top N recommendations in terms of the original item ids
top_N_items_df = get_df_rows_from_ids(top_N, "id", df_matrix_mf)

top_N_items_df[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
8010,michelle,TOMORROW X TOGETHER,Chasing That Feeling,2023,84,0.690,0.842,0.0739,0.01530,0.000000,0.0639,0.6260,141.994,-4.288,182670,2023,84
8011,michelle,TOMORROW X TOGETHER,Back for More (TXT Ver.),2023,72,0.735,0.938,0.1130,0.01560,0.000000,0.2350,0.8680,115.881,-3.448,162834,2023,72
8013,michelle,TOMORROW X TOGETHER,Dreamer,2023,74,0.532,0.573,0.0473,0.21300,0.000000,0.1100,0.2640,159.090,-6.183,186584,2023,74
8014,michelle,TOMORROW X TOGETHER,Happily Ever After,2023,77,0.688,0.683,0.0932,0.01700,0.000000,0.1340,0.5280,156.087,-6.629,151918,2023,77
8016,michelle,TOMORROW X TOGETHER,Deep Down,2023,72,0.801,0.471,0.0795,0.05970,0.000253,0.0675,0.4100,123.987,-9.083,163619,2023,72
8023,michelle,Mitski,Washing Machine Heart,2018,86,0.613,0.822,0.0260,0.00920,0.506000,0.4290,0.5670,105.997,-4.087,128198,2018,86
8054,michelle,TOMORROW X TOGETHER,Chasing That Feeling,2023,84,0.690,0.842,0.0739,0.01530,0.000000,0.0639,0.6260,141.994,-4.288,182670,2023,84
8055,michelle,TOMORROW X TOGETHER,Back for More (TXT Ver.),2023,72,0.735,0.938,0.1130,0.01560,0.000000,0.2350,0.8680,115.881,-3.448,162834,2023,72
8056,michelle,TOMORROW X TOGETHER,Dreamer,2023,74,0.532,0.573,0.0473,0.21300,0.000000,0.1100,0.2640,159.090,-6.183,186584,2023,74
8057,michelle,TOMORROW X TOGETHER,Happily Ever After,2023,77,0.688,0.683,0.0932,0.01700,0.000000,0.1340,0.5280,156.087,-6.629,151918,2023,77
